chapter-1 covers six metrics (BLEU, ROUGE, METEOR, BERTScore, MAUVE, Perplexity) on their own

chapter-1-llms-classical-approaches.ipynb — Classical NLP Evaluation Metrics (reference implementation)
Implements and compares six classical text-generation metrics (BLEU, ROUGE-1/2/L, METEOR, BERTScore, MAUVE, Perplexity), with interpretation notes and a comparison chart. Run on illustrative example sentences to validate correct metric computation. Status: real, executed code; toy data, not a benchmark run.

### 1: Installation & Setup

In [2]:
# Install required packages
!pip install evaluate nltk bert-score mauve-text -q

# Import all libraries
import os
import nltk
import torch
from evaluate import load
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
import mauve
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("✅ Setup complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.6 MB/s eta 0:00:00:00:0100:01


2026-01-25 14:13:08.766870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769350389.012324      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769350389.083218      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769350389.660945      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769350389.661007      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769350389.661011      55 computation_placer.cc:177] computation placer alr

✅ Setup complete!


### 2: Sample Data Preparation

In [ ]:
# Define sample references and candidates
# Replace these with your actual LLM outputs

references = [
    "the cat sat on the mat",
    "machine learning is a powerful tool for data analysis",
    "the quick brown fox jumps over the lazy dog"
]

candidates = [
    "the cat is sitting on the mat",
    "ML is very powerful for analyzing data",
    "a quick brown fox jumped over a lazy dog"
]

# For MAUVE (corpus-level comparison)
human_texts = [
    "the cat sat on the mat",
    "machine learning is a powerful tool for data analysis",
    "the quick brown fox jumps over the lazy dog",
    "artificial intelligence is transforming the world",
    "natural language processing helps computers understand text"
]

model_texts = [
    "the cat is sitting on the mat",
    "ML is very powerful for analyzing data",
    "a quick brown fox jumped over a lazy dog",
    "AI is changing how we live and work",
    "NLP enables machines to comprehend human language"
]

print(f"📊 Data loaded: {len(candidates)} candidate texts")
print(f"📊 Reference texts: {len(references)}")
print(f"📊 Human corpus: {len(human_texts)} texts")
print(f"📊 Model corpus: {len(model_texts)} texts")

### 3: BLEU Score

In [ ]:
print("=" * 60)
print("METRIC 1: BLEU SCORE")
print("=" * 60)

# Method 1: Using NLTK (sentence-level)
def compute_bleu_nltk(candidate, reference):
    ref_tokens = [reference.split()]
    cand_tokens = candidate.split()
    smoothie = SmoothingFunction().method4
    return sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smoothie)

# Method 2: Using Hugging Face evaluate (corpus-level)
bleu_metric = load("bleu")
bleu_results = bleu_metric.compute(
    predictions=candidates, 
    references=[[ref] for ref in references]
)

print(f"\n📈 Corpus-level BLEU: {bleu_results['bleu']:.4f}")
print(f"📊 Precisions: {bleu_results['precisions']}")

# Individual scores
print("\n📋 Individual BLEU scores:")
for i, (cand, ref) in enumerate(zip(candidates, references)):
    score = compute_bleu_nltk(cand, ref)
    print(f"  Sample {i+1}: {score:.4f}")

### 4: ROUGE Score

In [ ]:
print("=" * 60)
print("METRIC 2: ROUGE SCORE")
print("=" * 60)

rouge_metric = load('rouge')
rouge_results = rouge_metric.compute(
    predictions=candidates,
    references=references,
    use_aggregator=True,
    use_stemmer=True
)

print("\n📈 ROUGE Scores:")
print(f"  ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"  ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"  ROUGE-L: {rouge_results['rougeL']:.4f}")
print(f"  ROUGE-Lsum: {rouge_results['rougeLsum']:.4f}")

print("\n📝 Interpretation:")
print("  ROUGE-1: Unigram overlap")
print("  ROUGE-2: Bigram overlap")
print("  ROUGE-L: Longest common subsequence")
print("  ROUGE-Lsum: LCS on summary level")

### 5: METEOR Score

In [ ]:
print("=" * 60)
print("METRIC 3: METEOR SCORE")
print("=" * 60)

def compute_meteor(candidates, references):
    scores = []
    for cand, ref in zip(candidates, references):
        score = meteor_score([ref.split()], cand.split())
        scores.append(score)
    return scores

meteor_scores = compute_meteor(candidates, references)
avg_meteor = sum(meteor_scores) / len(meteor_scores)

print(f"\n📈 Average METEOR Score: {avg_meteor:.4f}")

print("\n📋 Individual METEOR scores:")
for i, score in enumerate(meteor_scores):
    print(f"  Sample {i+1}: {score:.4f}")

print("\n📝 Note: METEOR considers synonyms and stemming")

### 6: BERTScore

In [ ]:
print("=" * 60)
print("METRIC 4: BERTScore")
print("=" * 60)

# Using a lighter model for Kaggle (deberta-xlarge might be too large)
P, R, F1 = bert_score(
    cands=candidates,
    refs=references,
    lang="en",
    model_type="microsoft/deberta-base-mnli",  # Lighter model for Kaggle
    verbose=False
)

print(f"\n📈 BERTScore Results:")
print(f"  Precision: {P.mean().item():.4f}")
print(f"  Recall: {R.mean().item():.4f}")
print(f"  F1: {F1.mean().item():.4f}")

print("\n📋 Individual F1 scores:")
for i, f1_score in enumerate(F1):
    print(f"  Sample {i+1}: {f1_score.item():.4f}")

print("\n📝 Note: Using contextual embeddings for semantic similarity")

### 7: MAUVE Score

In [ ]:
print("=" * 60)
print("METRIC 5: MAUVE SCORE")
print("=" * 60)

# MAUVE compares two text distributions
mauve_result = mauve.compute_mauve(
    p_text=human_texts,  # Human-written texts
    q_text=model_texts,   # Model-generated texts
    device_id=-1,  # Use CPU (-1) or GPU (0) if available
    max_text_length=512,
    featurize_model_name="gpt2",
    verbose=False
)

print(f"\n📈 MAUVE Score: {mauve_result.mauve:.4f}")
print(f"📊 Divergence Curve: {len(mauve_result.divergence_curve)} points")

print("\n📝 Interpretation:")
print("  Range: 0 to 1 (higher is better)")
print("  Measures distributional similarity between human and model text")
print("  Does NOT require paired references!")

### 8: Perplexity (Optional - Requires GPU)

In [ ]:
print("=" * 60)
print("METRIC 6: PERPLEXITY")
print("=" * 60)

# WARNING: This requires significant memory and GPU
# Use a small model for demonstration

try:
    model_name = "gpt2"  # Small model for Kaggle
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    if torch.cuda.is_available():
        model = model.to("cuda")
    
    def compute_perplexity(texts):
        total_loss = 0
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            if torch.cuda.is_available():
                inputs = {k: v.to("cuda") for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs, labels=inputs["input_ids"])
                total_loss += outputs.loss.item()
        
        avg_loss = total_loss / len(texts)
        perplexity = torch.exp(torch.tensor(avg_loss))
        return perplexity.item()
    
    ppl = compute_perplexity(candidates)
    print(f"\n📈 Average Perplexity: {ppl:.2f}")
    print(f"\n📝 Note: Lower perplexity = better fluency")
    print(f"⚠️ Only valid when scored with the SAME model that generated the text!")
    
except Exception as e:
    print(f"\n⚠️ Perplexity calculation skipped (requires GPU/memory)")
    print(f"Error: {str(e)}")

### 9: Summary & Comparison

In [ ]:
print("=" * 70)
print("SUMMARY: ALL METRICS")
print("=" * 70)

print("\n📊 Results Summary:")
print(f"  BLEU:       {bleu_results['bleu']:.4f}")
print(f"  ROUGE-1:    {rouge_results['rouge1']:.4f}")
print(f"  ROUGE-L:    {rouge_results['rougeL']:.4f}")
print(f"  METEOR:     {avg_meteor:.4f}")
print(f"  BERTScore:  {F1.mean().item():.4f}")
print(f"  MAUVE:      {mauve_result.mauve:.4f}")

print("\n" + "=" * 70)
print("📝 METRIC CHARACTERISTICS")
print("=" * 70)
print("""
BLEU:       ✓ N-gram precision  ✗ Recall issues  → Good for translation
ROUGE:      ✓ Recall-focused    ✓ Multiple variants → Good for summarization  
METEOR:     ✓ Synonyms/stems    ✓ Balanced F1 → Better than BLEU
BERTScore:  ✓ Semantic matching ✓ Contextual → Best correlation with humans
MAUVE:      ✓ No references!    ✓ Distribution-level → Good for generation
Perplexity: ✓ Fluency          ✗ Model-dependent → Use with caution
""")

### 10: Visualization (Optional)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create comparison visualization
metrics_names = ['BLEU', 'ROUGE-1', 'ROUGE-L', 'METEOR', 'BERTScore', 'MAUVE']
metrics_values = [
    bleu_results['bleu'],
    rouge_results['rouge1'],
    rouge_results['rougeL'],
    avg_meteor,
    F1.mean().item(),
    mauve_result.mauve
]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics_names, metrics_values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b'])
plt.ylabel('Score', fontsize=12)
plt.title('Classical Metrics Comparison', fontsize=14, fontweight='bold')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars, metrics_values):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{value:.3f}',
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()